# 🧠 Conversational Agent with Memory

**Build an AI agent that remembers you!**

This notebook demonstrates Praval's memory system by creating a conversational agent that:
- 💾 **Remembers** previous conversations
- 🎯 **Asks relevant** follow-up questions based on what it knows
- 🤝 **Responds personally** using knowledge from earlier interactions
- 🔍 **Searches semantically** across all stored memories

**No setup required!** The memory system uses ChromaDB embedded storage - everything works immediately.

## 📦 Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / '.env')

from praval import agent, chat, start_agents

print("✅ Imports successful!")
print("✅ Memory system initialized (ChromaDB embedded)")
print("\n🎉 This agent will remember our conversations!")

## 🤖 Create Memory-Enabled Agent

Let's create a conversational assistant that uses memory to provide personalized responses.

In [ ]:
@agent("conversational_assistant", 
       memory=True,  # Enable memory!
       responds_to=["user_message"])
def memory_assistant(spore):
    """
    I'm a conversational assistant with memory.
    I remember our past conversations and use that context 
    to provide personalized, relevant responses.
    """
    user_input = spore.knowledge.get("message")
    
    print(f"\n💬 You: {user_input}")
    print("\n🔍 Searching memories...")
    
    # 1. RECALL relevant past conversations
    past_context = memory_assistant.recall(user_input, limit=5)
    
    # 2. Build context string from memories
    if past_context:
        print(f"✅ Found {len(past_context)} relevant memories!\n")
        context_summary = "What I remember about you:\n"
        for i, memory in enumerate(past_context, 1):
            context_summary += f"{i}. {memory.content[:100]}...\n"
            print(f"   📝 Memory {i}: {memory.content[:80]}...")
    else:
        print("🆕 This is our first conversation!\n")
        context_summary = "This is our first conversation!"
    
    print("\n🤔 Generating response...")
    
    # 3. GENERATE response using memory context
    response = chat(f"""
    You are a helpful conversational assistant with memory.
    
    {context_summary}
    
    User's current message: {user_input}
    
    Instructions:
    - Reference relevant memories in your response
    - Ask thoughtful follow-up questions based on what you know
    - Be warm and personalized
    - Show that you remember the conversation history
    - Keep responses concise (2-3 sentences)
    """)
    
    # 4. REMEMBER this interaction
    memory_text = f"User said: {user_input}. I responded about: {response[:100]}"
    memory_assistant.remember(memory_text, importance=0.7)
    
    print(f"\n🤖 Assistant: {response}")
    print(f"\n💾 Stored this interaction in memory")
    print("="*70)
    
    return {"response": response, "memories_used": len(past_context)}

print("✅ Agent created successfully!")
print("\n🎯 The agent has three memory superpowers:")
print("   1. 🔍 Recalls relevant past conversations")
print("   2. 🧠 Uses that context to respond personally")
print("   3. 💾 Stores new interactions for future use")

## 💬 Let's Have Conversations!

Watch how the agent remembers context and builds on previous conversations.

### Helper Function

In [ ]:
def talk(message):
    """Helper function to make conversations easy"""
    result = start_agents(
        memory_assistant,
        initial_data={
            "type": "user_message",
            "message": message
        }
    )
    return result

print("✅ Helper function ready!")
print("\nUse: talk('your message here')")

### 🎬 Conversation 1: Introduction

Let's start by introducing ourselves. The agent will remember this!

In [ ]:
talk("Hi! I'm learning Python for web development.")

### 💡 Conversation 2: Building on Context

Now let's ask for advice. Watch how the agent remembers we're learning Python for web dev!

In [ ]:
talk("What should I learn next?")

### 🎯 Conversation 3: Adding More Context

Let's mention a specific interest. The agent will combine this with previous memories.

In [ ]:
talk("I'm particularly interested in building REST APIs.")

### 🚀 Conversation 4: Technical Question

Now ask a technical question. The agent should recall your entire learning journey!

In [ ]:
talk("Which Python web framework should I use?")

### 🔮 Conversation 5: Memory Test

Let's explicitly ask the agent what it remembers!

In [ ]:
talk("Tell me what you remember about me and my learning goals.")

## 🎮 Your Turn!

Try your own conversations below. The agent will remember everything!

In [ ]:
# Try your own message!
talk("What does an ORM do in Django? How do I use one effectively?")

In [ ]:
# Another conversation
talk("Are there similar ones in other frameworks such as FastAPI?")

In [ ]:
# Keep going!
talk("What have you learned about me so far? Can you summarize, from our conversation? Can you give me something really interesting and zen based on what you know?")

## 🔍 Memory Inspection

Let's peek inside the agent's memory to see what it has stored!

### View All Stored Memories

In [ ]:
# Retrieve all memories (empty query returns all)
all_memories = memory_assistant.recall("", limit=50)

print(f"📚 Total memories stored: {len(all_memories)}")
print("\n" + "="*70)

for i, mem in enumerate(all_memories, 1):
    print(f"\n{i}. {mem.content}")
    print(f"   ⏰ Created: {mem.created_at.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   ⭐ Importance: {mem.importance}")
    print(f"   👁️  Accessed: {mem.access_count} times")

### Semantic Search

Search memories by meaning, not just keywords!

In [ ]:
# Search for specific topics
search_query = "learning programming"

print(f"🔎 Searching for: '{search_query}'")
print("="*70)

results = memory_assistant.recall(search_query, limit=5)

print(f"\n✅ Found {len(results)} relevant memories:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. {result.content}")
    print(f"   📊 Relevance: {result.importance}")
    print()

### Try Your Own Searches

In [ ]:
# Search for something specific
custom_search = "web development"  # Change this!

results = memory_assistant.recall(custom_search, limit=5)

print(f"🔎 Searching for: '{custom_search}'")
print(f"✅ Found {len(results)} memories\n")

for i, result in enumerate(results, 1):
    print(f"{i}. {result.content[:100]}...")

## 📊 Memory Statistics

In [ ]:
# Get conversation context
try:
    context_turns = memory_assistant.get_conversation_context(turns=20)
    
    print("📈 Memory System Statistics")
    print("="*70)
    print(f"💬 Total conversation turns: {len(context_turns)}")
    print(f"📚 Total memories: {len(all_memories)}")
    print(f"⭐ Average importance: {sum(m.importance for m in all_memories) / len(all_memories):.2f}")
    print(f"👁️  Total memory accesses: {sum(m.access_count for m in all_memories)}")
    
    print("\n🎯 Most accessed memories:")
    sorted_by_access = sorted(all_memories, key=lambda m: m.access_count, reverse=True)[:3]
    for i, mem in enumerate(sorted_by_access, 1):
        print(f"  {i}. ({mem.access_count}x) {mem.content[:60]}...")
        
except Exception as e:
    print(f"Note: Some memory statistics may not be available: {e}")
    print(f"\nBasic stats:")
    print(f"📚 Total memories: {len(all_memories)}")

## 🎓 What We Learned

### Memory Capabilities Demonstrated:

1. **✅ Automatic Storage** 
   - Agent automatically stores each interaction
   - No manual memory management needed

2. **✅ Semantic Search**
   - Search by meaning, not just keywords
   - Finds conceptually related memories

3. **✅ Context Awareness**
   - Agent references previous conversations
   - Builds on earlier topics naturally

4. **✅ Personalized Responses**
   - Remembers user preferences and goals
   - Adapts follow-up questions accordingly

5. **✅ Zero Configuration**
   - ChromaDB embedded storage
   - Works immediately, no setup required

### The Agent's Memory Workflow:

```
User Input → Search Memories → Build Context → Generate Response → Store Interaction
     ↑                                                                      ↓
     └──────────────────── Memory Persists ──────────────────────────────┘
```

## 🚀 Next Steps

### Experiment Further:

1. **Test Memory Persistence**
   - Close this notebook and reopen it
   - The agent should still remember your conversations!

2. **Try Different Topics**
   - Talk about hobbies, projects, goals
   - See how the agent builds a profile of you

3. **Explore Memory Search**
   - Search for different topics
   - Notice how semantic search finds related concepts

4. **Build Your Own Agent**
   - Copy the agent pattern
   - Customize for your use case
   - Add domain-specific knowledge

### Key Praval Memory APIs:

```python
# Enable memory in agent
@agent("name", memory=True)

# Store a memory
agent.remember("important fact", importance=0.8)

# Search memories
results = agent.recall("query", limit=5)

# Get specific memory
memory = agent.recall_by_id("id")

# Get conversation history
context = agent.get_conversation_context(turns=10)
```

### Learn More:

- 📖 [Memory System Documentation](../docs/memory-system.md)
- 🎯 [Multi-Agent Memory Example](../examples/005_memory_enabled_agents.py)
- 🌐 [Praval GitHub](https://github.com/aiexplorations/praval)